# QAG on the WACV hybrid: Quantum Attention Gate on CVC-ClinicDB

**Host model:** WACV hybrid, MK-UNet + UltraLight PVM (Mamba) inserted in 8 stages (encoder2-5, decoder1-4), alpha differential LR.
**Change under test:** the Grouped Attention Gate on skip connections (AG1..AG4).

Set `GATE_MODE` in the CONFIG cell, one notebook run per experiment:

| Run | GATE_MODE | What |
|---|---|---|
| E0 | `gag` | original classical gate (baseline) |
| E1 | `none` | gate removed |
| E2 | `replace` | quantum circuit replaces the g+x fusion |
| E3 | `classical` | same pipeline, circuit swapped for a same-param MLP |
| E4 | `augment` | classical logit + alpha * quantum logit |

Everything else (split, seed, recipe, eval) is identical across runs.
**Split:** EMCAD's official ClinicDB split (pre-split train/val/test folders from the EMCAD repo). **Recipe:** WACV 2027 submission (150 epochs, 256, no augmentation).
**Run order inside a session:** run every cell top to bottom. Cell 7c (smoke test + T4 benchmark) prints a time estimate. Check it before committing Kaggle hours.


In [ ]:
# ============================================================
# Cell 0a — Pin this notebook to a specific physical GPU
# ============================================================
# CRITICAL: must run BEFORE any `import torch` anywhere in the notebook.

import os
PHYSICAL_GPU_ID = 0
os.environ["CUDA_VISIBLE_DEVICES"] = str(PHYSICAL_GPU_ID)
print(f"Pinned: this notebook will see only physical GPU {PHYSICAL_GPU_ID}")


In [ ]:
# ============================================================
# Cell 1 — Environment check (verifies GPU pin worked)
# ============================================================
import torch
print(f"PyTorch              : {torch.__version__}")
print(f"CUDA                 : {torch.version.cuda}")
print(f"GPUs visible to torch: {torch.cuda.device_count()}  (should be 1)")
if torch.cuda.is_available():
    print(f"Device 0             : {torch.cuda.get_device_name(0)}")
else:
    print("⚠ CUDA NOT AVAILABLE — check Kaggle accelerator dropdown")
!python --version


In [ ]:
# ============================================================
# Cell 2 — Imports, cuDNN benchmark (mamba-ssm + timm + thop)
# ============================================================
# Note: timm is required for MK-UNet's weight init helpers
#       (trunc_normal_tf_ and named_apply).
#       mamba-ssm provides the Mamba block used inside the UltraLight PVM Layer.
#       The seed block has moved to the CONFIG cell so RUN_NUMBER controls it.

import os, json, csv, random, time, glob, shutil, math
from functools import partial
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# timm — needed for MK_UNet weight init (trunc_normal_tf_, named_apply)
try:
    import timm
    from timm.models.layers import trunc_normal_tf_
    from timm.models.helpers import named_apply
    print(f"✓ timm {timm.__version__}")
except ImportError:
    print("Installing timm...")
    !pip install -q timm
    from timm.models.layers import trunc_normal_tf_
    from timm.models.helpers import named_apply

# mamba-ssm + causal-conv1d — required for the PVM bottleneck (EXP6+).
# Use the prebuilt wheels matched to this Kaggle environment
# (cu12 / torch 2.10 / cxx11abi=True / cp312). These avoid on-the-fly
# CUDA compilation, which is fragile on Kaggle.
try:
    import causal_conv1d
    from mamba_ssm import Mamba
    print(f"✓ causal-conv1d {causal_conv1d.__version__}")
    print(f"✓ mamba-ssm imported")
except ImportError:
    print("Installing causal-conv1d + mamba-ssm from prebuilt wheels...")
    !pip install -q --no-deps --timeout 120 --retries 5 https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
    !pip install -q --no-deps --timeout 120 --retries 5 https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
    import causal_conv1d
    from mamba_ssm import Mamba
    print(f"✓ causal-conv1d {causal_conv1d.__version__}")
    print(f"✓ mamba-ssm installed")

# Quick smoke test (verifies the CUDA kernel actually runs)
_m = Mamba(d_model=64, d_state=16).cuda()
_x = torch.randn(1, 100, 64).cuda()
print(f"✓ mamba-ssm — forward pass OK: {_m(_x).shape}")
del _m, _x
torch.cuda.empty_cache()

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except ImportError:
    print("Installing thop...")
    !pip install -q thop
    from thop import profile as thop_profile
    THOP_AVAILABLE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}  (= physical GPU {PHYSICAL_GPU_ID})")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False
print(f"cuDNN benchmark : {torch.backends.cudnn.benchmark}")

# SEED is applied in the CONFIG cell (Cell 3), where RUN_NUMBER is defined.


In [ ]:
# ============================================================
# Cell 3 — CONFIG (edit this cell only)
# ============================================================
# MK-UNet (paper-strict, arXiv:2509.18493) + UltraLight VM-UNet PVMLayer
# (arXiv:2403.20035) inserted into 8 stages of the U-Net
# (encoder2-5, decoder1-4). This is the WACV HYBRID.
#
# Two architecture modes (see hybrid_mode):
#   "replace" — each target stage's MKIR block is REPLACED by a PVMLayer.
#   "insert"  — a PVMLayer is INSERTED after the stage's existing MKIR block.
#
# Two differential-LR partitions (see lr_mode):
#   "alpha" — fast/UltraLight group = PVMLayer params only.
#   "beta"  — fast/UltraLight group = ALL params in the 8 modified stages.
#
# Training uses a TWO-GROUP AdamW:
#   • UltraLight group : lr=1e-3, wd=1e-2, cosine schedule (T_max, eta_min below)
#   • MK-UNet group    : lr=1e-4, wd=1e-4, CONSTANT (no scheduler)
# Eval protocol and the model's logits output are byte-identical to EXP1/EXP8,
# so results compare directly against those baselines.

# ── Multi-run identity ──────────────────────────────────────────
# Bump RUN_NUMBER for each independent run (1, 2, 3, ...). The output slug
# carries the run number so runs never overwrite each other.
#
# SEED is held CONSTANT across runs (supervisor instruction: same seed for
# consistency). All seed-controlled randomness — weight init, dataloader
# shuffle order — is identical between runs. Variance between runs therefore
# comes only from hardware nondeterminism: cuDNN benchmark-mode algorithm
# selection, non-deterministic CUDA atomics, and AMP/GradScaler fp16 ops.
RUN_NUMBER = 1                       # ◄ edit for each independent run
SEED       = 42                      # constant across all runs (Sir's instruction)
VARY_INIT_SEED = False               # True -> init seed = SEED + RUN_NUMBER - 1
                                     # (split stays fixed either way). Ask Sir:
                                     # a random circuit init is part of the variance.
INIT_SEED  = SEED + RUN_NUMBER - 1 if VARY_INIT_SEED else SEED

# ── Gate under test (the ONLY thing that changes between E0..E4) ─
GATE_MODE     = "gag"                # "gag" | "none" | "replace" | "classical" | "augment"
GATE_TARGETS  = ["AG1"]              # AG1 = deepest skip (16x16 @256). Later: ["AG1","AG2"], ...
QAG_LAYERS    = 3                    # circuit depth (data re-uploading layers)
GATES_FAST_LR = True                 # target gates join the fast (UL) LR group in EVERY
                                     # mode, so gate recipes match across E0..E4.

# ── What this experiment is ─────────────────────────────────────
EXPERIMENT_ID    = "QAG"
EXPERIMENT_TAG   = f"{GATE_MODE}_{'-'.join(GATE_TARGETS)}_L{QAG_LAYERS}_ClinicDB"
IN_CHANNELS      = 3              # RGB only
SPATIAL_ENCODING = "none"        # no coordinate channels in this variant

# ── Hybrid placement + differential-LR mode ─────────────────────
HYBRID_MODE = "insert"  # "replace" | "insert"  (LOCKED for this notebook)
LR_MODE     = "alpha"      # "alpha"   | "beta"     (LOCKED for this notebook)

# ── UltraLight PVMLayer (Mamba) hyperparameters ─────────────────
PVM_D_STATE      = 16             # Mamba SSM state size (UltraLight default)
PVM_D_CONV       = 4              # Mamba 1D conv kernel (UltraLight default)
PVM_EXPAND       = 2              # Mamba expansion factor (UltraLight default)

# ── Differential learning-rate recipe ───────────────────────────
# Fast group = UltraLight/PVM. Slow group = MK-UNet backbone.
UL_LR            = 1e-3           # UltraLight group base LR
UL_WD            = 1e-2           # UltraLight group weight decay
UL_COSINE_TMAX   = 50             # cosine T_max for UL group (see note below)
UL_COSINE_ETAMIN = 1e-5           # cosine eta_min for UL group
MK_LR            = 1e-4           # MK-UNet group LR (paper-strict, CONSTANT)
MK_WD            = 1e-4           # MK-UNet group weight decay (paper-strict)
# NOTE on UL_COSINE_TMAX: WACV recipe, kept as-is (T_max=50, 150 epochs).
# The cosine is PERIODIC: the UL LR decays 1e-3 -> 1e-5 over epochs 0-50, rises
# back to 1e-3 by epoch 100, and decays again to 1e-5 at epoch 150. So training
# ends exactly at eta_min (150 = 3 x 50). The target gates share this schedule.

# ── Dataset path (Kaggle input) ─────────────────────────────────
# EMCAD's official ClinicDB split (Sir's instruction: do NOT make our own).
# Source: EMCAD repo README -> "Polyp datasets" Google Drive -> polyp/target/ClinicDB
# Expected layout (same as EMCAD train_polyp.py):
#   <root>/train/images  <root>/train/masks
#   <root>/val/images    <root>/val/masks
#   <root>/test/images   <root>/test/masks
# Images and masks are matched by file stem.
CLINICDB_ROOT = "/kaggle/input/datasets/moon1570/clinicdb/ClinicDB"

# ── Resume flags ────────────────────────────────────────────────
RESUME = False
KAGGLE_MODEL_INPUT_DIR = None

# ── Paths under /kaggle/working ─────────────────────────────────
KAGGLE_WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs_local"
SLUG = f"{EXPERIMENT_ID.lower()}_{EXPERIMENT_TAG.lower()}_run{RUN_NUMBER}"

CONFIG = {
    # Experiment identity
    "experiment_id"    : EXPERIMENT_ID,
    "experiment_name"  : f"{EXPERIMENT_ID}_{EXPERIMENT_TAG}_run{RUN_NUMBER}",
    "description"      : f"WACV hybrid (MK-UNet + PVM 8 stages) with gate_mode={GATE_MODE} on {GATE_TARGETS} - CVC-ClinicDB, RGB",
    "slug"             : SLUG,
    "run_number"       : RUN_NUMBER,
    "seed"             : SEED,
    "init_seed"        : INIT_SEED,
    "vary_init_seed"   : VARY_INIT_SEED,
    "spatial_encoding" : SPATIAL_ENCODING,

    # Data
    "dataset"          : "CVC-ClinicDB",
    "clinicdb_root"    : CLINICDB_ROOT,
    "split_source"     : "EMCAD official (pre-split folders)",
    "img_size"      : 256,            # same as WACV runs
    "in_channels"   : IN_CHANNELS,
    "num_classes"   : 1,

    # MK-UNet architecture — paper-strict
    "mk_channels"          : [16, 32, 64, 96, 160],
    "mk_depths"            : [1, 1, 1, 1, 1],
    "mk_kernel_sizes"      : [1, 3, 5],
    "mk_expansion_factor"  : 2,
    "mk_gag_kernel"        : 3,

    # Hybrid PVM placement + LR partition
    "hybrid_mode"       : HYBRID_MODE,
    "lr_mode"           : LR_MODE,
    "pvm_d_state"       : PVM_D_STATE,
    "pvm_d_conv"        : PVM_D_CONV,
    "pvm_expand"        : PVM_EXPAND,

    # Gate under test
    "gate_mode"         : GATE_MODE,
    "gate_targets"      : GATE_TARGETS,
    "qag_layers"        : QAG_LAYERS,
    "gates_fast_lr"     : GATES_FAST_LR,

    # Differential LR — UltraLight (fast) group
    "ul_lr"             : UL_LR,
    "ul_wd"             : UL_WD,
    "ul_cosine_tmax"    : UL_COSINE_TMAX,
    "ul_cosine_eta_min" : UL_COSINE_ETAMIN,
    # Differential LR — MK-UNet (slow) group
    "mk_lr"             : MK_LR,
    "mk_wd"             : MK_WD,

    # Training — paper-strict (except the dual-group optimizer above)
    "batch_size"        : 16,
    "epochs"            : 200,
    "grad_clip"         : 0.5,        # paper Sec 4.2
    "scheduler"         : "cosine_ul_only",  # cosine on UL group; MK group constant
    "augmentation"      : False,      # paper: "We do not apply any form of augmentation"
    "multiscale_train"  : True,
    "multiscale_rates"  : [0.75, 1.0, 1.25],

    # Loss — BCE + Dice (UltraLight VM-UNet's BceDiceLoss, 1:1).
    # Model returns RAW LOGITS; the loss applies sigmoid internally in fp32.
    "loss"              : "bce_dice",

    # Eval — paper-strict (resize to original, min-max norm, per-image Dice)
    "eval_protocol"     : "paper_strict",

    # Hardware
    "num_workers"     : 4,
    "pin_memory"      : True,
    "prefetch_factor" : 4,

    # Output paths
    "output_dir"     : os.path.join(KAGGLE_WORKING, "outputs",     SLUG),
    "checkpoint_dir" : os.path.join(KAGGLE_WORKING, "checkpoints", SLUG),
    "log_dir"        : os.path.join(KAGGLE_WORKING, "logs",        SLUG),
}

# Apply the seed AFTER CONFIG.
random.seed(INIT_SEED); np.random.seed(INIT_SEED)
torch.manual_seed(INIT_SEED); torch.cuda.manual_seed_all(INIT_SEED)

for d in [CONFIG["output_dir"], CONFIG["checkpoint_dir"], CONFIG["log_dir"]]:
    os.makedirs(d, exist_ok=True)

with open(os.path.join(CONFIG["output_dir"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=4)

print(f"=== {CONFIG['experiment_name']} ===")
print(f"  hybrid_mode        : {CONFIG['hybrid_mode']}   (PVM in encoder2-5 + decoder1-4)")
print(f"  lr_mode            : {CONFIG['lr_mode']}")
print(f"  RUN_NUMBER         : {CONFIG['run_number']}    (slug suffix _run{CONFIG['run_number']})")
print(f"  SEED               : {CONFIG['seed']}   (constant across runs — supervisor instruction)")
print(f"  in_channels        : {CONFIG['in_channels']}   (spatial_encoding={CONFIG['spatial_encoding']})")
print(f"  UL group LR/WD     : {CONFIG['ul_lr']} / {CONFIG['ul_wd']}   (cosine T_max={CONFIG['ul_cosine_tmax']}, eta_min={CONFIG['ul_cosine_eta_min']})")
print(f"  MK group LR/WD     : {CONFIG['mk_lr']} / {CONFIG['mk_wd']}   (constant, no scheduler)")
print(f"  loss               : {CONFIG['loss']}   (model emits logits; sigmoid inside loss)")
print(f"  gate_mode          : {GATE_MODE}  targets={GATE_TARGETS}  layers={QAG_LAYERS}  fast_lr={GATES_FAST_LR}")
print(f"  init seed          : {INIT_SEED}  (vary={VARY_INIT_SEED})   split: EMCAD official")
for _sp in ("train", "val", "test"):
    for _sub in ("images", "masks"):
        _p = os.path.join(CLINICDB_ROOT, _sp, _sub)
        print(f"  {_sp+'/'+_sub:<18} : exists={os.path.isdir(_p)}  {_p}")
if not os.path.isdir(os.path.join(CLINICDB_ROOT, "train", "images")):
    print("  ⚠️  Set CLINICDB_ROOT to the EMCAD ClinicDB folder before running.")


In [ ]:
# ============================================================
# Cell 4 — Spatial coordinate helpers (X, Y, radial)
# ============================================================
# Unused in EXP1 (spatial_encoding="none"). Defined here for the v2 series.
#
# Conventions (Nathan & Kansal "Lesion-Net" 2020):
#   X      : column position, normalized to [0, 1] left → right
#   Y      : row position,    normalized to [0, 1] top  → bottom
#   radial : Euclidean distance from centre, normalized to [0, 1]
#            r(x, y) = sqrt((x-0.5)^2 + (y-0.5)^2) / sqrt(0.5)

def generate_x_coord(height, width):
    x_lin = np.linspace(0, 1, width, dtype=np.float32)
    return np.broadcast_to(x_lin, (height, width)).copy()

def generate_y_coord(height, width):
    y_lin = np.linspace(0, 1, height, dtype=np.float32).reshape(-1, 1)
    return np.broadcast_to(y_lin, (height, width)).copy()

def generate_radial_coord(height, width):
    x = np.linspace(0, 1, width,  dtype=np.float32)
    y = np.linspace(0, 1, height, dtype=np.float32)
    xx, yy = np.meshgrid(x, y)
    r = np.sqrt((xx - 0.5)**2 + (yy - 0.5)**2) / np.sqrt(0.5)
    return r.astype(np.float32)


def generate_spatial_channels(height, width, encoding):
    encoding = encoding.lower()
    if encoding == "none":
        return None
    if encoding == "x":
        chans = [generate_x_coord(height, width)]
    elif encoding == "y":
        chans = [generate_y_coord(height, width)]
    elif encoding == "xy":
        chans = [generate_x_coord(height, width), generate_y_coord(height, width)]
    elif encoding == "xyz":
        chans = [generate_x_coord(height, width),
                 generate_y_coord(height, width),
                 generate_radial_coord(height, width)]
    else:
        raise ValueError(f"Unknown spatial encoding: {encoding!r}")
    return np.stack(chans, axis=0)


print("Spatial coordinate helpers defined.")


In [ ]:
# ============================================================
# Cell 5 — CVC-ClinicDB (EMCAD official split) + dataset
# ============================================================
# Uses EMCAD's pre-split folders so our numbers use the same split as the
# lab's published ClinicDB results. No split is created here.
# NOTE: ClinicDB frames come from 31 videos. A frame-level split puts
# near-duplicate frames in train and test, which inflates scores. Report this.
# Returns (img, mask, [orig_H, orig_W]), same contract as the ISIC dataset.

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def load_emcad_split(root, out_json):
    """Read <root>/{train,val,test}/{images,masks}; pair by file stem."""
    split = {"source": "EMCAD official", "root": root}
    for sp in ("train", "val", "test"):
        idir, mdir = os.path.join(root, sp, "images"), os.path.join(root, sp, "masks")
        imgs  = {os.path.splitext(f)[0]: f for f in os.listdir(idir) if f.lower().endswith(IMG_EXTS)}
        masks = {os.path.splitext(f)[0]: f for f in os.listdir(mdir) if f.lower().endswith(IMG_EXTS)}
        unpaired = set(imgs) ^ set(masks)
        assert not unpaired, f"{sp}: unpaired files {sorted(unpaired)[:10]}"
        stems = sorted(imgs, key=lambda s: (len(s), s))
        split[sp] = [[imgs[k], masks[k]] for k in stems]
    n = {sp: len(split[sp]) for sp in ("train", "val", "test")}
    tot = sum(n.values())
    print(f"EMCAD split: train {n['train']} | val {n['val']} | test {n['test']}  (total {tot})")
    if tot != 612:
        print(f"  ⚠️  expected 612 ClinicDB frames, found {tot}. Check the folder.")
    overlap = set(map(tuple, split["train"])) & set(map(tuple, split["test"]))
    assert not overlap, f"train/test overlap: {list(overlap)[:5]}"
    with open(out_json, "w") as f:
        json.dump(split, f, indent=2)          # record of exactly what was used
    print(f"  file list saved -> {out_json}")
    return split


class ClinicDBDataset(Dataset):
    def __init__(self, pairs, img_dir, mask_dir, img_size=256, spatial_encoding="none"):
        self.pairs, self.img_dir, self.mask_dir = pairs, img_dir, mask_dir
        self.spatial_encoding = spatial_encoding.lower()
        # Paper-strict: NO augmentation (multi-scale resize happens in the train loop)
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_f, mask_f = self.pairs[idx]
        img_pil = Image.open(os.path.join(self.img_dir, img_f)).convert("RGB")
        orig_H, orig_W = img_pil.size[1], img_pil.size[0]
        img  = np.array(img_pil)
        mask = np.array(Image.open(os.path.join(self.mask_dir, mask_f)).convert("L"))
        aug  = self.transform(image=img, mask=mask)
        img, mask = aug["image"], (aug["mask"] > 127).float()
        if self.spatial_encoding != "none":
            H, W = img.shape[1], img.shape[2]
            img = torch.cat([img, torch.from_numpy(
                generate_spatial_channels(H, W, self.spatial_encoding))], 0)
        return img, mask.unsqueeze(0), torch.tensor([orig_H, orig_W], dtype=torch.long)

print("ClinicDB split + dataset defined.")


In [ ]:
# ============================================================
# Cell 7 — Paper's MK_UNet architecture (verbatim) + UltraLight PVMLayer
# ============================================================
# MK_UNet code copied directly from the official mkunet_network.py without
# modification, EXCEPT for one surgical change: __init__ now accepts an
# optional bottleneck_module that runs after encoder5's max-pool and before
# CA1+SA. When bottleneck_module is None (the default), the architecture is
# byte-for-byte identical to the paper. EXP6+ instantiate it with a PVMLayer.
#
# PVMLayer (the parallel vision mamba layer) is from:
#   Wu et al., "UltraLight VM-UNet: Parallel Vision Mamba significantly reduces
#   parameters for skin lesion segmentation", arXiv:2403.20035 (2024).
# The implementation matches the authors\' official code: LayerNorm,
# 4-way channel split, a SHARED Mamba block applied to each split,
# concat, second LayerNorm, linear projection, with a learnable scalar
# skip connection from the input.

def gcd(a, b):
    while b:
        a, b = b, a % b
    return a


def _init_weights(module, name, scheme=''):
    if isinstance(module, nn.Conv2d):
        if scheme == 'normal':
            nn.init.normal_(module.weight, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'trunc_normal':
            trunc_normal_tf_(module.weight, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'xavier_normal':
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'kaiming_normal':
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        else:
            fan_out = module.kernel_size[0] * module.kernel_size[1] * module.out_channels
            fan_out //= module.groups
            nn.init.normal_(module.weight, 0, math.sqrt(2.0 / fan_out))
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.LayerNorm):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)


def act_layer(act, inplace=False, neg_slope=0.2, n_prelu=1):
    act = act.lower()
    if   act == 'relu':      return nn.ReLU(inplace)
    elif act == 'relu6':     return nn.ReLU6(inplace)
    elif act == 'leakyrelu': return nn.LeakyReLU(neg_slope, inplace)
    elif act == 'prelu':     return nn.PReLU(num_parameters=n_prelu, init=neg_slope)
    elif act == 'gelu':      return nn.GELU()
    elif act == 'hswish':    return nn.Hardswish(inplace)
    else: raise NotImplementedError(f'activation layer [{act}] is not found')


def channel_shuffle(x, groups):
    batchsize, num_channels, height, width = x.data.size()
    channels_per_group = num_channels // groups
    x = x.view(batchsize, groups, channels_per_group, height, width)
    x = torch.transpose(x, 1, 2).contiguous()
    x = x.view(batchsize, -1, height, width)
    return x


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, out_planes=None, ratio=16, activation='relu'):
        super().__init__()
        self.in_planes = in_planes
        self.out_planes = out_planes
        if self.in_planes < ratio:
            ratio = self.in_planes
        self.reduced_channels = self.in_planes // ratio
        if self.out_planes is None:
            self.out_planes = in_planes
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.activation = act_layer(activation, inplace=True)
        self.fc1 = nn.Conv2d(in_planes, self.reduced_channels, 1, bias=False)
        self.fc2 = nn.Conv2d(self.reduced_channels, self.out_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.init_weights('normal')

    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        avg_out = self.fc2(self.activation(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.activation(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size in (3, 7, 11)
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.init_weights('normal')

    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)


class GroupedAttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int, kernel_size=1, groups=1, activation='relu'):
        super().__init__()
        if kernel_size == 1:
            groups = 1
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=kernel_size, stride=1,
                      padding=kernel_size // 2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=kernel_size, stride=1,
                      padding=kernel_size // 2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid())
        self.activation = act_layer(activation, inplace=True)
        self.init_weights('normal')

    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.activation(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class MultiKernelDepthwiseConv(nn.Module):
    def __init__(self, in_channels, kernel_sizes, stride, activation='relu6', dw_parallel=True):
        super().__init__()
        self.in_channels = in_channels
        self.dw_parallel = dw_parallel
        self.dwconvs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels, k, stride, k // 2,
                          groups=in_channels, bias=False),
                nn.BatchNorm2d(in_channels),
                act_layer(activation, inplace=True),
            )
            for k in kernel_sizes
        ])
        self.init_weights('normal')

    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        outputs = []
        for dwconv in self.dwconvs:
            dw_out = dwconv(x)
            outputs.append(dw_out)
            if not self.dw_parallel:
                x = x + dw_out
        return outputs


class MultiKernelInvertedResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride, expansion_factor=2,
                 dw_parallel=True, add=True, kernel_sizes=[1, 3, 5], activation='relu6'):
        super().__init__()
        assert stride in [1, 2]
        self.stride = stride
        self.in_c = in_c
        self.out_c = out_c
        self.kernel_sizes = kernel_sizes
        self.add = add
        self.n_scales = len(kernel_sizes)
        self.use_skip_connection = (stride == 1)
        self.ex_c = int(in_c * expansion_factor)
        self.pconv1 = nn.Sequential(
            nn.Conv2d(in_c, self.ex_c, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ex_c),
            act_layer(activation, inplace=True),
        )
        self.multi_scale_dwconv = MultiKernelDepthwiseConv(
            self.ex_c, kernel_sizes, stride, activation, dw_parallel=dw_parallel)
        if add:
            self.combined_channels = self.ex_c
        else:
            self.combined_channels = self.ex_c * self.n_scales
        self.pconv2 = nn.Sequential(
            nn.Conv2d(self.combined_channels, out_c, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_c),
        )
        if self.use_skip_connection and (in_c != out_c):
            self.conv1x1 = nn.Conv2d(in_c, out_c, 1, 1, 0, bias=False)
        self.init_weights('normal')

    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        pout1 = self.pconv1(x)
        dwconv_outs = self.multi_scale_dwconv(pout1)
        if self.add:
            dout = 0
            for d in dwconv_outs:
                dout = dout + d
        else:
            dout = torch.cat(dwconv_outs, dim=1)
        dout = channel_shuffle(dout, gcd(self.combined_channels, self.out_c))
        out = self.pconv2(dout)
        if self.use_skip_connection:
            if self.in_c != self.out_c:
                x = self.conv1x1(x)
            return x + out
        else:
            return out


def mk_irb_bottleneck(in_c, out_c, n, s, expansion_factor=2,
                     dw_parallel=True, add=True, kernel_sizes=[1, 3, 5], activation='relu6'):
    convs = [MultiKernelInvertedResidualBlock(
        in_c, out_c, s, expansion_factor=expansion_factor,
        dw_parallel=dw_parallel, add=add, kernel_sizes=kernel_sizes, activation=activation)]
    for _ in range(n - 1):
        convs.append(MultiKernelInvertedResidualBlock(
            out_c, out_c, 1, expansion_factor=expansion_factor,
            dw_parallel=dw_parallel, add=add, kernel_sizes=kernel_sizes, activation=activation))
    return nn.Sequential(*convs)


# ────────────────────────────────────────────────────────────────────
# UltraLight VM-UNet — PVM Layer
# ────────────────────────────────────────────────────────────────────
# Source: Wu et al., "UltraLight VM-UNet: Parallel Vision Mamba significantly
#         reduces parameters for skin lesion segmentation", arXiv:2403.20035 (2024).
# Implementation follows the authors\' official PVMLayer:
#   - input (B, C, H, W) flattened to (B, N=H*W, C)
#   - LayerNorm
#   - split C into 4 equal groups along the channel dim
#   - apply the SAME Mamba block to each group in parallel
#     (this is what makes the layer "ultralight" — Mamba weights are reused)
#   - learnable scalar skip from the input to each Mamba output
#   - concat the 4 outputs along C
#   - LayerNorm + Linear projection
#   - reshape back to (B, output_dim, H, W)
# Requires input_dim divisible by 4. At MK-UNet\'s deepest level C=160 → 40/group.

class PVMLayer(nn.Module):
    def __init__(self, input_dim, output_dim=None, d_state=16, d_conv=4, expand=2):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim
        assert input_dim % 4 == 0, f"PVMLayer: input_dim={input_dim} must be divisible by 4"
        self.input_dim  = input_dim
        self.output_dim = output_dim
        self.norm  = nn.LayerNorm(input_dim)
        self.mamba = Mamba(
            d_model = input_dim // 4,   # shared Mamba on 1/4 channels
            d_state = d_state,
            d_conv  = d_conv,
            expand  = expand,
        )
        self.proj       = nn.Linear(input_dim, output_dim)
        self.skip_scale = nn.Parameter(torch.ones(1))

    def forward(self, x):
        # Mamba\'s selective scan kernel can be unstable in fp16; cast to fp32.
        # autocast will downcast for subsequent ops as needed.
        if x.dtype == torch.float16:
            x = x.to(torch.float32)
        B, C = x.shape[:2]
        assert C == self.input_dim, f"PVMLayer: expected C={self.input_dim}, got C={C}"
        n_tokens = x.shape[2:].numel()
        img_dims = x.shape[2:]

        x_flat = x.reshape(B, C, n_tokens).transpose(-1, -2)   # (B, N, C)
        x_norm = self.norm(x_flat)
        x1, x2, x3, x4 = torch.chunk(x_norm, 4, dim=2)          # 4 × (B, N, C/4)

        # Shared Mamba applied 4 times in parallel; learnable skip on each branch.
        x_mamba1 = self.mamba(x1) + self.skip_scale * x1
        x_mamba2 = self.mamba(x2) + self.skip_scale * x2
        x_mamba3 = self.mamba(x3) + self.skip_scale * x3
        x_mamba4 = self.mamba(x4) + self.skip_scale * x4

        x_mamba = torch.cat([x_mamba1, x_mamba2, x_mamba3, x_mamba4], dim=2)  # (B, N, C)
        x_mamba = self.norm(x_mamba)
        x_mamba = self.proj(x_mamba)                            # (B, N, C_out)
        return x_mamba.transpose(-1, -2).reshape(B, self.output_dim, *img_dims)


class MK_UNet(nn.Module):
    """
    Paper\'s MK_UNet architecture (5-stage encoder + 5-stage decoder).
    Default channels [16, 32, 64, 96, 160] → 0.316M params, 0.314 GFLOPs (256x256).
    For binary segmentation, returns [p4] (single output, full resolution).
    For multi-class, returns [p4, p3, p2, p1] for deep supervision.

    SURGICAL MODIFICATION FOR EXP6+ (Arc B):
        bottleneck_module — optional nn.Module applied to the deepest feature
        map (after encoder5\'s max_pool, BEFORE CA1+SA). When None (default),
        the architecture is byte-for-byte identical to the paper. When a
        PVMLayer is passed in, it runs first so Mamba sees raw deepest
        features and CA1+SA then filter the Mamba output. This placement
        is LOCKED per the project handoff.
    """
    def __init__(self, num_classes=1, in_channels=3,
                 channels=[16, 32, 64, 96, 160], depths=[1, 1, 1, 1, 1],
                 kernel_sizes=[1, 3, 5], expansion_factor=2, gag_kernel=3,
                 deep_supervision=False,
                 bottleneck_module=None,   # ← new for EXP6+ (None = paper-strict)
                 **kwargs):
        super().__init__()
        self.deep_supervision = deep_supervision

        # Encoder (5 stages)
        self.encoder1 = mk_irb_bottleneck(in_channels, channels[0], depths[0], 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.encoder2 = mk_irb_bottleneck(channels[0], channels[1], depths[1], 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.encoder3 = mk_irb_bottleneck(channels[1], channels[2], depths[2], 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.encoder4 = mk_irb_bottleneck(channels[2], channels[3], depths[3], 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.encoder5 = mk_irb_bottleneck(channels[3], channels[4], depths[4], 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)

        # Optional deepest-level bottleneck (Identity in paper-strict EXP1-5).
        self.bottleneck_module = bottleneck_module if bottleneck_module is not None else nn.Identity()

        # Grouped Attention Gates
        self.AG1 = GroupedAttentionGate(F_g=channels[3], F_l=channels[3], F_int=channels[3] // 2,
                                        kernel_size=gag_kernel, groups=channels[3] // 2)
        self.AG2 = GroupedAttentionGate(F_g=channels[2], F_l=channels[2], F_int=channels[2] // 2,
                                        kernel_size=gag_kernel, groups=channels[2] // 2)
        self.AG3 = GroupedAttentionGate(F_g=channels[1], F_l=channels[1], F_int=channels[1] // 2,
                                        kernel_size=gag_kernel, groups=channels[1] // 2)
        self.AG4 = GroupedAttentionGate(F_g=channels[0], F_l=channels[0], F_int=channels[0] // 2,
                                        kernel_size=gag_kernel, groups=channels[0] // 2)

        # Decoder (5 stages)
        self.decoder1 = mk_irb_bottleneck(channels[4], channels[3], 1, 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.decoder2 = mk_irb_bottleneck(channels[3], channels[2], 1, 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.decoder3 = mk_irb_bottleneck(channels[2], channels[1], 1, 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.decoder4 = mk_irb_bottleneck(channels[1], channels[0], 1, 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)
        self.decoder5 = mk_irb_bottleneck(channels[0], channels[0], 1, 1,
                                          expansion_factor=expansion_factor, kernel_sizes=kernel_sizes)

        # Channel Attention per decoder stage
        self.CA1 = ChannelAttention(channels[4], ratio=16)
        self.CA2 = ChannelAttention(channels[3], ratio=16)
        self.CA3 = ChannelAttention(channels[2], ratio=16)
        self.CA4 = ChannelAttention(channels[1], ratio=8)
        self.CA5 = ChannelAttention(channels[0], ratio=4)

        # Shared spatial attention
        self.SA = SpatialAttention()

        # Segmentation heads
        self.out1 = nn.Conv2d(channels[2], num_classes, kernel_size=1)
        self.out2 = nn.Conv2d(channels[1], num_classes, kernel_size=1)
        self.out3 = nn.Conv2d(channels[0], num_classes, kernel_size=1)
        self.out4 = nn.Conv2d(channels[0], num_classes, kernel_size=1)

    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        # Encoder
        out = F.max_pool2d(self.encoder1(x), 2, 2); t1 = out
        out = F.max_pool2d(self.encoder2(out), 2, 2); t2 = out
        out = F.max_pool2d(self.encoder3(out), 2, 2); t3 = out
        out = F.max_pool2d(self.encoder4(out), 2, 2); t4 = out
        out = F.max_pool2d(self.encoder5(out), 2, 2)

        # ─── Deepest-level bottleneck (Identity in paper-strict EXP1-5;
        #     UltraLight PVMLayer in EXP6+). Placement is LOCKED: before CA1+SA.
        out = self.bottleneck_module(out)

        # Decoder Stage 4 (no aux head)
        out = self.CA1(out) * out
        out = self.SA(out) * out
        out = F.relu(F.interpolate(self.decoder1(out), scale_factor=2, mode='bilinear', align_corners=False))
        t4 = self.AG1(g=out, x=t4)
        out = torch.add(out, t4)

        # Decoder Stage 3 → aux head p1 (low-res)
        out = self.CA2(out) * out
        out = self.SA(out) * out
        out = F.relu(F.interpolate(self.decoder2(out), scale_factor=2, mode='bilinear', align_corners=False))
        p1 = F.interpolate(self.out1(out), scale_factor=8, mode='bilinear', align_corners=False)
        t3 = self.AG2(g=out, x=t3)
        out = torch.add(out, t3)

        # Decoder Stage 2 → aux head p2
        out = self.CA3(out) * out
        out = self.SA(out) * out
        out = F.relu(F.interpolate(self.decoder3(out), scale_factor=2, mode='bilinear', align_corners=False))
        p2 = F.interpolate(self.out2(out), scale_factor=4, mode='bilinear', align_corners=False)
        t2 = self.AG3(g=out, x=t2)
        out = torch.add(out, t2)

        # Decoder Stage 1 → aux head p3
        out = self.CA4(out) * out
        out = self.SA(out) * out
        out = F.relu(F.interpolate(self.decoder4(out), scale_factor=2, mode='bilinear', align_corners=False))
        p3 = F.interpolate(self.out3(out), scale_factor=2, mode='bilinear', align_corners=False)
        t1 = self.AG4(g=out, x=t1)
        out = torch.add(out, t1)

        # Final Stage → main head p4 (full resolution)
        out = self.CA5(out) * out
        out = self.SA(out) * out
        out = F.relu(F.interpolate(self.decoder5(out), scale_factor=2, mode='bilinear', align_corners=False))
        p4 = self.out4(out)

        if self.deep_supervision:
            return [p4, p3, p2, p1]
        return [p4]   # paper\'s binary protocol: only main head


def make_pvm_bottleneck(channels, cfg):
    """Build the UltraLight PVM Layer at the deepest level."""
    return PVMLayer(
        input_dim = channels[-1],          # 160 by default
        output_dim= channels[-1],
        d_state   = cfg["pvm_d_state"],
        d_conv    = cfg["pvm_d_conv"],
        expand    = cfg["pvm_expand"],
    )

# ────────────────────────────────────────────────────────────────────
# HYBRID PVM-IN-STAGES (new for EXP10-17)
# ────────────────────────────────────────────────────────────────────
# WACV placement: UltraLight PVM Layer added to the stages in HYBRID_STAGES
# (encoder2-5, decoder1-4 = 8 stages). Two architecture styles:
#
#   REPLACE : the stage's MKIR block is removed; a PVMLayer takes its place,
#             mapping the same (in_c -> out_c). No CNN left in that stage.
#   INSERT  : the stage's MKIR block is KEPT and a PVMLayer (out_c -> out_c)
#             is appended after it  ->  nn.Sequential(MKIR, PVM).
#
# This is done as a POST-CONSTRUCTION surgery on a paper-strict MK_UNet so the
# locked MK_UNet class is never edited. bottleneck_module stays Identity for
# the hybrid (the deepest-bottleneck PVM of EXP6/7 is a DIFFERENT placement and
# is NOT used here).
#
# Stage in/out channels for channels=[c0,c1,c2,c3,c4]:
#   encoder2 : c0 -> c1   encoder3 : c1 -> c2   encoder4 : c2 -> c3   encoder5 : c3 -> c4
#   decoder1 : c4 -> c3   decoder2 : c3 -> c2   decoder3 : c2 -> c1   decoder4 : c1 -> c0
# INSERT uses PVM(out_c -> out_c); every out_c in [16,32,64,96,160] is divisible
# by 4, so PVMLayer's 4-way channel split is valid at every modified stage.

HYBRID_STAGES = ["encoder2", "encoder3", "encoder4", "encoder5",
                 "decoder1", "decoder2", "decoder3", "decoder4"]


def _stage_io_channels(channels, in_channels):
    """(in_c, out_c) for every modified stage. Ablation: a PVM is inserted into the stages
    named in HYBRID_STAGES. out_c is what INSERT mode uses (PVM(out_c->out_c)); every
    out_c below is divisible by 4, so the PVMLayer channel split is valid."""
    c0, c1, c2, c3, c4 = channels
    return {
        "encoder1": (in_channels, c0),
        "encoder2": (c0, c1),
        "encoder3": (c1, c2),
        "encoder4": (c2, c3),
        "encoder5": (c3, c4),
        "decoder1": (c4, c3),
        "decoder2": (c3, c2),
        "decoder3": (c2, c1),
        "decoder4": (c1, c0),
        "decoder5": (c0, c0),
    }


def build_stage_pvm(in_c, out_c, cfg):
    """A PVMLayer sized for one modified stage."""
    assert in_c % 4 == 0, f"PVM stage in_c={in_c} must be divisible by 4"
    return PVMLayer(
        input_dim = in_c,
        output_dim= out_c,
        d_state   = cfg["pvm_d_state"],
        d_conv    = cfg["pvm_d_conv"],
        expand    = cfg["pvm_expand"],
    )


def apply_pvm_hybrid(model, mode, channels, cfg):
    """Surgically add PVM to every stage in HYBRID_STAGES (8 stages here).

    mode == "replace": stage = PVMLayer(in_c -> out_c)            [MKIR removed]
    mode == "insert" : stage = Sequential(original_MKIR, PVMLayer(out_c -> out_c))

    Returns the same model (modified in place). The swapped modules are tracked
    in model._pvm_hybrid_info for reporting.
    """
    assert mode in ("replace", "insert"), f"unknown hybrid mode {mode!r}"
    io = _stage_io_channels(channels, cfg["in_channels"])
    info = {"mode": mode, "stages": {}}

    for name in HYBRID_STAGES:
        in_c, out_c = io[name]
        original = getattr(model, name)

        if mode == "replace":
            new_stage = build_stage_pvm(in_c, out_c, cfg)              # in_c -> out_c
        else:  # insert: keep MKIR (in_c->out_c), append PVM (out_c->out_c)
            pvm = build_stage_pvm(out_c, out_c, cfg)
            new_stage = nn.Sequential(original, pvm)

        setattr(model, name, new_stage)
        info["stages"][name] = {"in_c": in_c, "out_c": out_c}

    model._pvm_hybrid_info = info
    return model


def pvm_parameters(model):
    """All parameters belonging to PVMLayer submodules (for the alpha LR group)."""
    ids, params = set(), []
    for m in model.modules():
        if isinstance(m, PVMLayer):
            for p in m.parameters(recurse=True):
                if id(p) not in ids:
                    ids.add(id(p)); params.append(p)
    return params


def modified_stage_parameters(model, stages=HYBRID_STAGES):
    """All parameters inside the modified stages (for the beta LR group).

    For REPLACE these are exactly the PVM params (the stage IS a PVMLayer);
    for INSERT they additionally include the kept MKIR params.
    """
    ids, params = set(), []
    for name in stages:
        stage = getattr(model, name)
        for p in stage.parameters(recurse=True):
            if id(p) not in ids:
                ids.add(id(p)); params.append(p)
    return params


def split_param_groups(model, lr_mode):
    """Partition model parameters into (ul_params, mk_params).

    lr_mode == "alpha": ul = PVMLayer params only.
    lr_mode == "beta" : ul = all params in the modified stages.
    The two lists are guaranteed disjoint and to cover every parameter exactly
    once (asserted), so AdamW receives a clean partition.
    """
    assert lr_mode in ("alpha", "beta"), f"unknown lr_mode {lr_mode!r}"
    ul = pvm_parameters(model) if lr_mode == "alpha" else modified_stage_parameters(model)
    ul_ids = {id(p) for p in ul}
    mk = [p for p in model.parameters() if id(p) not in ul_ids]

    n_ul = sum(p.numel() for p in ul)
    n_mk = sum(p.numel() for p in mk)
    n_total = sum(p.numel() for p in model.parameters())
    assert n_ul + n_mk == n_total, (
        f"param partition leak: ul({n_ul}) + mk({n_mk}) != total({n_total})")
    assert len(ul) + len(mk) == len(list(model.parameters())), "param tensor count mismatch"
    return ul, mk, n_ul, n_mk

print("MK_UNet (paper-strict) + PVMLayer + hybrid surgery (REPLACE/INSERT) defined.")
print("  HYBRID_STAGES:", HYBRID_STAGES)

In [ ]:
# ============================================================
# Cell 7b — Quantum Attention Gate (QAG) + gate surgery + build_model()
# ============================================================
# Swaps MK-UNet's GroupedAttentionGate(s) (AG1..AG4) post-construction, like
# apply_pvm_hybrid(). MK_UNet itself is never edited.
#
# Classical GAG : psi = sigmoid(BN(conv1x1( relu(W_g(g) + W_x(x)) )))
# Quantum  QAG  : psi = sigmoid(BN(conv1x1( Circuit(W_g(g), W_x(x)) )))
#   -> the circuit replaces the "add + ReLU" fusion. Per pixel, 2 angles from
#      the skip x go to qubits 0,1 and 2 angles from the gating signal g go to
#      qubits 2,3. They mix ONLY through entanglement (CNOT ring).
#   Each layer: data re-uploading RY -> trainable RY,RZ -> CNOT ring 0>1>2>3>0.
#   Readout <Z> on 4 qubits -> 4 channels in [-1,1].
#
# Simulator: exact 4-qubit statevector in plain PyTorch (16 complex amplitudes
# per pixel, batched over B*H*W), gradients by autograd. Always fp32 (AMP-safe).
# Math verified against PennyLane default.qubit (max abs diff ~1e-15).
#
# GATE_MODE (set in CONFIG cell):
#   "gag"       original classical GAG           (E0 baseline)
#   "none"      gate removed, skip passes as-is  (E1)
#   "replace"   quantum circuit replaces fusion  (E2)
#   "classical" same pipeline, circuit -> tiny MLP with ~same params (E3)
#   "augment"   classical logit + alpha*quantum logit, alpha init 0 (E4)

N_QUBITS = 4

def _init_state(n_batch, device):
    s = torch.zeros(n_batch, 2 ** N_QUBITS, dtype=torch.complex64, device=device)
    s[:, 0] = 1.0
    return s.view(n_batch, *([2] * N_QUBITS))

def _apply_1q(state, U, q):
    s = state.movedim(q + 1, -1)
    if U.dim() == 2:
        s = torch.einsum("...j,ij->...i", s, U)
    else:
        s = torch.einsum("n...j,nij->n...i", s, U)
    return s.movedim(-1, q + 1)

def _apply_cnot(state, c, t):
    s0 = state.select(c + 1, 0)
    s1 = state.select(c + 1, 1)
    t_ax = t + 1 if t < c else t
    return torch.stack([s0, s1.flip(t_ax)], dim=c + 1)

def _ry(theta):
    c, s = torch.cos(theta / 2), torch.sin(theta / 2)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(torch.complex64)

def _rz(phi):
    e_m = torch.polar(torch.ones_like(phi), -phi / 2)
    e_p = torch.polar(torch.ones_like(phi),  phi / 2)
    z = torch.zeros_like(e_m)
    return torch.stack([torch.stack([e_m, z], -1), torch.stack([z, e_p], -1)], -2)

def _expval_z(state):
    p = state.abs() ** 2
    out = []
    for q in range(N_QUBITS):
        pq = p.movedim(q + 1, 1).reshape(p.shape[0], 2, -1).sum(-1)
        out.append(pq[:, 0] - pq[:, 1])
    return torch.stack(out, dim=1)


class QuantumMixer(nn.Module):
    """4-qubit data re-uploading circuit. (N,4) angles -> (N,4) <Z>."""
    def __init__(self, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.theta = nn.Parameter(0.1 * torch.randn(n_layers, N_QUBITS))
        self.phi   = nn.Parameter(0.1 * torch.randn(n_layers, N_QUBITS))

    def forward(self, angles):
        angles = angles.float()
        state = _init_state(angles.shape[0], angles.device)
        for l in range(self.n_layers):
            for q in range(N_QUBITS):
                state = _apply_1q(state, _ry(angles[:, q]), q)
            for q in range(N_QUBITS):
                state = _apply_1q(state, _ry(self.theta[l, q]), q)
                state = _apply_1q(state, _rz(self.phi[l, q]), q)
            for q in range(N_QUBITS):
                state = _apply_cnot(state, q, (q + 1) % N_QUBITS)
        return _expval_z(state)


class ClassicalMixer(nn.Module):
    """Same-param classical stand-in: 4 -> 2 -> 4, tanh (22 params vs 24 at L=3)."""
    def __init__(self, hidden=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(N_QUBITS, hidden), nn.Tanh(),
                                 nn.Linear(hidden, N_QUBITS), nn.Tanh())
    def forward(self, angles):
        return self.net(angles.float())


class QuantumGroupedAttentionGate(nn.Module):
    """Drop-in for GroupedAttentionGate: forward(g, x) -> x * psi."""
    def __init__(self, F_g, F_l, F_int, kernel_size=3, groups=1,
                 mode="replace", n_layers=3, activation="relu"):
        super().__init__()
        assert mode in ("replace", "augment", "classical")
        self.mode = mode
        if kernel_size == 1:
            groups = 1
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size, 1, kernel_size // 2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size, 1, kernel_size // 2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int))
        self.to_ang_x = nn.Conv2d(F_int, 2, 1, bias=True)
        self.to_ang_g = nn.Conv2d(F_int, 2, 1, bias=True)
        self.mixer  = ClassicalMixer() if mode == "classical" else QuantumMixer(n_layers)
        self.q_head = nn.Conv2d(N_QUBITS, 1, 1, bias=True)
        if mode == "augment":
            self.activation = act_layer(activation, inplace=False)
            self.c_head = nn.Conv2d(F_int, 1, 1, bias=True)
            self.alpha  = nn.Parameter(torch.zeros(1))
        self.bn = nn.BatchNorm2d(1)

    def forward(self, g, x):
        g1, x1 = self.W_g(g), self.W_x(x)
        B, _, H, W = x1.shape
        with torch.autocast("cuda", enabled=False):
            ax = math.pi * torch.tanh(self.to_ang_x(x1.float()))
            ag = math.pi * torch.tanh(self.to_ang_g(g1.float()))
            ang = torch.cat([ax, ag], 1).permute(0, 2, 3, 1).reshape(-1, N_QUBITS)
            z = self.mixer(ang).reshape(B, H, W, N_QUBITS).permute(0, 3, 1, 2)
            logit = self.q_head(z)
            if self.mode == "augment":
                c_logit = self.c_head(self.activation(g1.float() + x1.float()))
                logit = c_logit + self.alpha * logit
            psi = torch.sigmoid(self.bn(logit))
        return x * psi.to(x.dtype)


class IdentityGate(nn.Module):
    """E1: gate removed, skip passes through unchanged."""
    def forward(self, g, x):
        return x


def apply_gates(model, channels, gag_kernel, mode, targets, n_layers):
    """Swap the named AG modules. mode 'gag' leaves them untouched."""
    ch_for = {"AG1": channels[3], "AG2": channels[2], "AG3": channels[1], "AG4": channels[0]}
    for name in targets:
        if mode == "gag":
            continue
        if mode == "none":
            setattr(model, name, IdentityGate())
            continue
        c = ch_for[name]
        setattr(model, name, QuantumGroupedAttentionGate(
            F_g=c, F_l=c, F_int=c // 2, kernel_size=gag_kernel, groups=c // 2,
            mode=mode, n_layers=n_layers))
    model._gate_info = {"mode": mode, "targets": list(targets), "n_layers": n_layers}
    return model


def build_model(cfg):
    """WACV hybrid (MK-UNet + PVM in 8 stages) + gate swap. Single source of truth."""
    m = MK_UNet(
        num_classes=cfg["num_classes"], in_channels=cfg["in_channels"],
        channels=cfg["mk_channels"], depths=cfg["mk_depths"],
        kernel_sizes=cfg["mk_kernel_sizes"], expansion_factor=cfg["mk_expansion_factor"],
        gag_kernel=cfg["mk_gag_kernel"], deep_supervision=False, bottleneck_module=None)
    m = apply_pvm_hybrid(m, cfg["hybrid_mode"], cfg["mk_channels"], cfg)
    m = apply_gates(m, cfg["mk_channels"], cfg["mk_gag_kernel"],
                    cfg["gate_mode"], cfg["gate_targets"], cfg["qag_layers"])
    return m


def gate_parameters(model, targets):
    ids, params = set(), []
    for name in targets:
        for p in getattr(model, name).parameters():
            if id(p) not in ids:
                ids.add(id(p)); params.append(p)
    return params


def count_mixer_params(model):
    return sum(p.numel() for mm in model.modules()
               if isinstance(mm, (QuantumMixer, ClassicalMixer)) for p in mm.parameters())


# Redefine pvm_parameters so split_param_groups('alpha') also puts the target
# gates in the FAST (UltraLight) group when CONFIG['gates_fast_lr'] is True.
# Applied to EVERY gate mode (gag, replace, classical, augment), so all runs
# train their gates with the same recipe -> fair comparison.
def pvm_parameters(model):
    ids, params = set(), []
    for mm in model.modules():
        if isinstance(mm, PVMLayer):
            for p in mm.parameters(recurse=True):
                if id(p) not in ids:
                    ids.add(id(p)); params.append(p)
    if CONFIG.get("gates_fast_lr", False):
        for p in gate_parameters(model, CONFIG["gate_targets"]):
            if id(p) not in ids:
                ids.add(id(p)); params.append(p)
    return params

print(f"QAG defined. gate_mode={CONFIG['gate_mode']}  targets={CONFIG['gate_targets']}  "
      f"layers={CONFIG['qag_layers']}  gates_fast_lr={CONFIG['gates_fast_lr']}")


In [ ]:
# ============================================================
# Cell 7c — Smoke test + GPU benchmark (RUN THIS BEFORE TRAINING)
# ============================================================
# 1) Simulator unit tests with known answers
# 2) Full model forward + backward at all 3 training scales under AMP
# 3) Gradient reaches the circuit
# 4) Times real training steps -> estimated epoch time and total hours

# ── 1. simulator unit tests ─────────────────────────────────────
_dev = DEVICE
_a = torch.tensor([0.3, 1.1, 2.0], device=_dev)
_s = _apply_1q(_init_state(3, _dev), _ry(_a), 0)
assert torch.allclose(_expval_z(_s)[:, 0], torch.cos(_a), atol=1e-5), "RY test failed"
_s = _apply_cnot(_apply_1q(_init_state(1, _dev), _ry(torch.tensor([math.pi], device=_dev)), 0), 0, 1)
assert torch.allclose(_expval_z(_s)[0], torch.tensor([-1., -1., 1., 1.], device=_dev), atol=1e-5), "CNOT test failed"
_s = _apply_cnot(_apply_1q(_init_state(1, _dev), _ry(torch.tensor([math.pi/2], device=_dev)), 0), 0, 1)
assert torch.allclose(_expval_z(_s)[0], torch.tensor([0., 0., 1., 1.], device=_dev), atol=1e-5), "Bell test failed"
print("✓ simulator unit tests passed (RY, CNOT, Bell state)")

# ── 2-3. model forward/backward at every scale ──────────────────
torch.manual_seed(INIT_SEED)
_m = build_model(CONFIG).to(DEVICE).train()
_opt = torch.optim.AdamW(_m.parameters(), lr=1e-4)
_scaler = torch.amp.GradScaler("cuda")
bs = CONFIG["batch_size"]
rates = CONFIG["multiscale_rates"] if CONFIG["multiscale_train"] else [1.0]
torch.cuda.reset_peak_memory_stats()
step_times = []
for rate in rates:
    S = int(round(CONFIG["img_size"] * rate / 32) * 32)
    x = torch.randn(bs, CONFIG["in_channels"], S, S, device=DEVICE)
    y = (torch.rand(bs, 1, S, S, device=DEVICE) > 0.5).float()
    for it in range(4):                      # 1 warm-up + 3 timed
        torch.cuda.synchronize(); t0 = time.time()
        _opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out = _m(x)[0]
        loss = criterion_smoke = F.binary_cross_entropy_with_logits(out.float(), y)
        _scaler.scale(loss).backward()
        if it == 0 and CONFIG["gate_mode"] in ("replace", "augment", "classical"):
            gmix = getattr(_m, CONFIG["gate_targets"][0]).mixer
            gp = next(gmix.parameters()).grad
            assert gp is not None and torch.isfinite(gp).all(), "no/invalid grad in mixer"
        _scaler.step(_opt); _scaler.update()
        torch.cuda.synchronize()
        if it > 0:
            step_times.append((rate, time.time() - t0))
    print(f"  scale {rate:<4} ({S}x{S}): out {tuple(out.shape)}  loss {loss.item():.4f}  OK")
if CONFIG["gate_mode"] in ("replace", "augment", "classical"):
    print("✓ gradients reach the gate mixer")

peak = torch.cuda.max_memory_allocated() / 1e9
per_batch = sum(t for _, t in step_times) / 3        # 3 timed iters per scale -> one batch over all scales
n_batches = int(612 * 0.8) // bs                     # EMCAD split is ~80% train
est_epoch = per_batch * n_batches + 15               # +~15s for val pass
print(f"\nGPU               : {torch.cuda.get_device_name(0)}")
print(f"Peak memory       : {peak:.2f} GB (batch {bs}, largest scale)")
print(f"Train step (all scales) : {per_batch:.2f} s per batch")
print(f"Estimated epoch   : ~{est_epoch:.0f} s  |  {CONFIG['epochs']} epochs ≈ {est_epoch*CONFIG['epochs']/3600:.1f} h")
print("  (estimate from random tensors, excludes data loading. Real epoch 1 time is logged below.)")

print(f"\nParams total      : {sum(p.numel() for p in _m.parameters()):,}")
print(f"Mixer params      : {count_mixer_params(_m)}  (quantum 24 at L=3 vs classical 22)")
del _m, _opt, _scaler, x, y, out
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# Cell 8 — Hybrid architecture sanity (params per group + per-stage PVM FLOPs)
# ============================================================
# Builds the hybrid (MK-UNet + PVM in 8 stages + gate swap) and reports:
#   - output shape (must be a single LOGITS tensor [p4])
#   - param breakdown: total / PVM / CNN, and the alpha & beta group sizes
#   - FLOPs: CNN backbone via thop (thop has no counter for mamba-ssm's
#     selective_scan, so PVM shows ~0 there) + per-stage PVM FLOPs analytical.
#
# Per-stage spatial token counts at 256x256 input:
#   encoder2 @ 128x128 = 16384   encoder3 @ 64x64 = 4096
#   encoder4 @  32x32  =  1024   encoder5 @ 16x16 =  256
#   decoder1 @   8x8   =    64   decoder2 @ 16x16 =  256
#   decoder3 @  32x32  =  1024   decoder4 @ 64x64 = 4096
# PVM input_dim per stage depends on style:
#   REPLACE -> stage in_c   |   INSERT -> stage out_c

print(f"Building HYBRID ({CONFIG['hybrid_mode'].upper()}) MK_UNet, "
      f"channels={CONFIG['mk_channels']}, in_channels={CONFIG['in_channels']}")

_m = build_model(CONFIG).to(DEVICE).eval()

_x = torch.randn(1, CONFIG["in_channels"], CONFIG["img_size"], CONFIG["img_size"]).to(DEVICE)
with torch.no_grad():
    _out = _m(_x)

total_params = sum(p.numel() for p in _m.parameters())
pvm_total    = sum(p.numel() for mm in _m.modules() if isinstance(mm, PVMLayer)
                            for p in mm.parameters())
cnn_total    = total_params - pvm_total
_ul_a, _mk_a, n_ul_a, n_mk_a = split_param_groups(_m, "alpha")
_ul_b, _mk_b, n_ul_b, n_mk_b = split_param_groups(_m, "beta")

print(f"\n  Input shape    : {tuple(_x.shape)}")
print(f"  Output (list)  : {len(_out)} tensor(s)  (expect 1 = [p4] logits)")
print(f"  Output[0] shape: {tuple(_out[0].shape)}")
print(f"  Total params   : {total_params:>10,}  ({total_params/1e6:.3f}M)")
print(f"    PVM params   : {pvm_total:>10,}  ({pvm_total/1e6:.3f}M)")
print(f"    CNN params   : {cnn_total:>10,}  ({cnn_total/1e6:.3f}M)  "
      f"(paper-strict MK-UNet = 315,566)")
gate_total = sum(p.numel() for p in gate_parameters(_m, CONFIG["gate_targets"]))
print(f"    Gate params  : {gate_total:>10,}  ({CONFIG['gate_mode']} on {CONFIG['gate_targets']}; "
      f"mixer {count_mixer_params(_m)})")
print(f"  alpha groups   : UL {n_ul_a:>8,} | MK {n_mk_a:>8,}   (gates in UL: {CONFIG['gates_fast_lr']})")
print(f"  beta  groups   : UL {n_ul_b:>8,} | MK {n_mk_b:>8,}")
if CONFIG["hybrid_mode"] == "replace":
    print(f"  (REPLACE: alpha == beta partition -> {n_ul_a == n_ul_b})")

# ── PVM FLOPs (analytical, per stage) ──────────────────────────
def pvm_flops_analytical(input_dim, n_tokens, d_state=16, d_conv=4, expand=2):
    """Closed-form FLOPs for one PVMLayer.forward (4-way shared Mamba)."""
    d_model = input_dim // 4
    d_inner = expand * d_model
    dt_rank = math.ceil(d_model / 16)
    inproj   = 2 * n_tokens * d_model * (d_inner * 2)
    conv1d_  = 2 * n_tokens * d_inner * d_conv
    xproj    = 2 * n_tokens * d_inner * (dt_rank + d_state * 2)
    dtproj   = 2 * n_tokens * dt_rank * d_inner
    sscan    = 9 * n_tokens * d_inner * d_state + 6 * n_tokens * d_inner
    outproj  = 2 * n_tokens * d_inner * d_model
    mamba_one   = inproj + conv1d_ + xproj + dtproj + sscan + outproj
    mamba_total = 4 * mamba_one
    norms      = 2 * (4 * n_tokens * input_dim)
    outer_proj = 2 * n_tokens * input_dim * input_dim
    skip_add   = n_tokens * input_dim
    return mamba_total + norms + outer_proj + skip_add

S = CONFIG["img_size"]
_io = _stage_io_channels(CONFIG["mk_channels"], CONFIG["in_channels"])
_tokens = {"encoder1": (S//1)**2,  "encoder2": (S//2)**2,  "encoder3": (S//4)**2,
           "encoder4": (S//8)**2,  "encoder5": (S//16)**2,
           "decoder1": (S//32)**2, "decoder2": (S//16)**2, "decoder3": (S//8)**2,
           "decoder4": (S//4)**2,  "decoder5": (S//2)**2}
print(f"\n  Per-stage PVM FLOPs (analytical):")
pvm_flops_total = 0
for name in HYBRID_STAGES:
    in_c, out_c = _io[name]
    pvm_in = in_c if CONFIG["hybrid_mode"] == "replace" else out_c
    nt = _tokens[name]
    f = pvm_flops_analytical(pvm_in, nt, CONFIG["pvm_d_state"],
                             CONFIG["pvm_d_conv"], CONFIG["pvm_expand"])
    pvm_flops_total += f
    print(f"    {name:9s}: dim={pvm_in:3d}  tokens={nt:5d}  -> {f/1e9:.4f} G")
print(f"    {'TOTAL PVM':9s}: {pvm_flops_total/1e9:.4f} G")

# ── CNN backbone FLOPs via thop (PVM counts ~0 there) ──────────
if THOP_AVAILABLE:
    try:
        _mc = _m
        cnn_flops, _ = thop_profile(_mc, inputs=(_x,), verbose=False)
        print(f"\n  CNN backbone FLOPs : {cnn_flops/1e9:.3f} G  (thop; PVM and circuit not counted)")
        print(f"  PVM FLOPs          : {pvm_flops_total/1e9:.4f} G  (analytical)")
        print(f"  Total FLOPs        : {(cnn_flops + pvm_flops_total)/1e9:.3f} G")
    except Exception as e:
        print(f"\n  ⚠ thop failed ({type(e).__name__}: {e}); reporting PVM FLOPs only.")
else:
    print("  ⚠ thop unavailable — skipping CNN FLOPs.")

del _m, _x, _out
torch.cuda.empty_cache()
print("\n✓ Hybrid architecture OK")

In [ ]:
# ============================================================
# Cell 9 — Loss (BceDiceLoss) + metrics
# ============================================================
# HYBRID loss (EXP10+): UltraLight VM-UNet's BceDiceLoss — BCE + Dice, equal
# 1:1 weighting, NO boundary weighting, NO deep supervision. This matches the
# UltraLight repo's actual behaviour (utils.py BceDiceLoss; engine.py applies
# the criterion once to the single main output).
#
# IMPORTANT — logits vs sigmoid (project decision, documented):
#   The UltraLight repo's BceDiceLoss takes a SIGMOID'd prediction. We keep the
#   model returning RAW LOGITS (return [p4], same convention as EXP1-9) and
#   apply the sigmoid INSIDE this loss instead. Why: it leaves the carefully
#   validated evaluate_paper_strict() byte-identical (it sigmoids logits), so
#   the hybrid's ISIC/PH2 numbers stay directly comparable to the EXP1 (0.8711)
#   and EXP8 (0.8642) baselines used by the gating rule. The handoff sanctions
#   either choice ("Either works; document the choice") — this is the choice.
#
# AMP note: torch.autocast forbids BCELoss/binary_cross_entropy under fp16, so
#   the loss casts logits to fp32 and the train loop calls it OUTSIDE autocast.

class BceDiceLoss(nn.Module):
    """BCE + Dice (1:1), taking LOGITS; sigmoid applied internally in fp32.
    Numerically identical to UltraLight's BceDiceLoss(wb=1, wd=1) evaluated on
    the equivalent sigmoid'd prediction (verified)."""
    def __init__(self, wb=1.0, wd=1.0):
        super().__init__()
        self.wb = wb
        self.wd = wd

    def forward(self, logits, target):
        logits = logits.float()                 # AMP-safe: BCE must run in fp32
        pred   = torch.sigmoid(logits)
        size   = pred.size(0)
        pred_   = pred.view(size, -1)
        target_ = target.view(size, -1).float()

        # BCE term (== nn.BCELoss on probabilities)
        bce = F.binary_cross_entropy(pred_, target_)

        # Dice term (== UltraLight DiceLoss, smooth=1, per-sample then mean)
        smooth = 1
        inter      = (pred_ * target_).sum(1)
        dice_score = (2 * inter + smooth) / (pred_.sum(1) + target_.sum(1) + smooth)
        dice       = 1 - dice_score.sum() / size

        return self.wb * bce + self.wd * dice


# Module-level criterion used by the training loop (Cell 10).
criterion = BceDiceLoss(wb=1.0, wd=1.0)


# ── Per-image metrics (paper-strict) — unchanged from EXP1-9 ──────────────
def dice_coefficient(predicted, labels):
    smooth = 1e-6
    p = predicted.contiguous().view(-1)
    l = labels.contiguous().view(-1)
    inter = (p * l).sum()
    total = p.sum() + l.sum()
    return (2. * inter + smooth) / (total + smooth)


def iou_score(predicted, labels):
    smooth = 1e-6
    p = predicted.contiguous().view(-1)
    l = labels.contiguous().view(-1)
    inter = (p * l).sum()
    union = p.sum() + l.sum() - inter
    return (inter + smooth) / (union + smooth)


def per_image_metrics(pred_binary, target_binary):
    """All 6 metrics from a single binary prediction + target.
    Both inputs must be float tensors of shape (H, W) in {0, 1}."""
    TP = (pred_binary * target_binary).sum().item()
    FP = (pred_binary * (1 - target_binary)).sum().item()
    FN = ((1 - pred_binary) * target_binary).sum().item()
    TN = ((1 - pred_binary) * (1 - target_binary)).sum().item()
    s = 1e-6
    return {
        "dice"        : (2*TP+s) / (2*TP+FP+FN+s),
        "iou"         : (TP+s)   / (TP+FP+FN+s),
        "sensitivity" : (TP+s)   / (TP+FN+s),
        "specificity" : (TN+s)   / (TN+FP+s),
        "precision"   : (TP+s)   / (TP+FP+s),
        "accuracy"    : (TP+TN+s)/ (TP+TN+FP+FN+s),
    }


print("BceDiceLoss (BCE+Dice 1:1, logits-in, sigmoid inside, fp32) + per-image metrics defined.")
print("  criterion =", criterion.__class__.__name__, "(wb=1.0, wd=1.0)")

In [ ]:
# ============================================================
# Cell 10 — Train / paper-strict eval functions
# ============================================================
# Train: multi-scale resize {0.75, 1.0, 1.25}, grad clip 0.5, BceDiceLoss on
#        the primary output (model returns [p4] = LOGITS). The loss is computed
#        OUTSIDE the autocast block on pred.float(): torch.autocast forbids
#        BCELoss under fp16, so the BCE term must run in fp32.
# Eval:  IDENTICAL to EXP1-9 (resize logits to original (H,W), sigmoid, per-image
#        min-max normalise, binarise at 0.5). Unchanged so numbers stay directly
#        comparable to the EXP1/EXP8 baselines.

def train_one_epoch(model, loader, optimizer, device, scaler, grad_clip,
                    multiscale_train, multiscale_rates, img_size):
    model.train()
    total_loss = 0.0
    n_steps = 0
    pbar = tqdm(loader, desc="  Train", leave=False, dynamic_ncols=True)
    for i, (imgs, masks, _shapes) in enumerate(pbar):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        rates = multiscale_rates if multiscale_train else [1.0]
        for rate in rates:
            if rate != 1.0:
                trainsize = int(round(img_size * rate / 32) * 32)
                imgs_r  = F.interpolate(imgs,  size=(trainsize, trainsize),
                                        mode='bilinear', align_corners=True)
                masks_r = F.interpolate(masks, size=(trainsize, trainsize),
                                        mode='nearest')
            else:
                imgs_r, masks_r = imgs, masks

            optimizer.zero_grad(set_to_none=True)
            # Forward under autocast (fp16); loss in fp32 OUTSIDE autocast.
            with torch.amp.autocast("cuda"):
                out = model(imgs_r)
                pred = out[0] if isinstance(out, list) else out   # raw LOGITS
            loss = criterion(pred.float(), masks_r)               # BceDiceLoss, fp32

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            scaler.step(optimizer)
            scaler.update()

            if rate == 1.0:
                total_loss += loss.item()
                n_steps += 1

        cur_lr_ul = optimizer.param_groups[0]["lr"]   # UltraLight-recipe group
        cur_lr_mk = optimizer.param_groups[1]["lr"]   # MK-UNet-recipe group
        mem_gb = torch.cuda.memory_allocated() / 1e9
        running = total_loss / max(n_steps, 1)
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg =f"{running:.4f}",
            ul  =f"{cur_lr_ul:.1e}",
            mk  =f"{cur_lr_mk:.1e}",
            mem =f"{mem_gb:.1f}GB",
        )
    pbar.close()
    return total_loss / max(n_steps, 1)


@torch.no_grad()
def evaluate_paper_strict(model, loader, device, desc="Eval"):
    """
    Paper-strict eval (UNCHANGED from EXP1-9 — model returns LOGITS):
      For each image:
        1. Forward at training resolution (256x256) -> logits
        2. Resize logits to original (H, W) bilinear
        3. Sigmoid
        4. Min-max normalize per image to [0, 1]
        5. Threshold at 0.5 -> binary prediction
        6. Resize GT to original (H, W) nearest; threshold at 0.2
        7. Per-image Dice / IoU / etc. Average across images.
    """
    model.eval()
    metrics_sum = {k: 0.0 for k in ["dice", "iou", "sensitivity",
                                     "specificity", "precision", "accuracy"]}
    n = 0
    preds_for_viz = []
    for imgs, masks, shapes in tqdm(loader, desc=desc, leave=False):
        imgs  = imgs.to(device)
        masks = masks.to(device).float()

        out = model(imgs)
        pred = out[0] if isinstance(out, list) else out   # (B, 1, H_train, W_train) LOGITS

        for b in range(imgs.size(0)):
            orig_H = int(shapes[b][0].item())
            orig_W = int(shapes[b][1].item())

            p = pred[b:b+1]
            p_resized = F.interpolate(p, size=(orig_H, orig_W),
                                       mode='bilinear', align_corners=False)
            p_resized = torch.sigmoid(p_resized).squeeze()
            p_min = p_resized.min()
            p_max = p_resized.max()
            p_resized = (p_resized - p_min) / (p_max - p_min + 1e-8)
            pred_binary = (p_resized >= 0.5).float()

            g = masks[b:b+1]
            g_resized = F.interpolate(g, size=(orig_H, orig_W),
                                       mode='nearest').squeeze()
            target_binary = (g_resized >= 0.2).float()

            m = per_image_metrics(pred_binary, target_binary)
            for k in metrics_sum:
                metrics_sum[k] += m[k]
            n += 1

            if len(preds_for_viz) < 8:
                p_viz = torch.sigmoid(pred[b]).cpu()
                preds_for_viz.append(p_viz)

    avg = {k: v / max(n, 1) for k, v in metrics_sum.items()}
    return avg, preds_for_viz


print("Train (BceDiceLoss, fp32 loss outside autocast) / eval (paper-strict, unchanged) defined.")

In [ ]:
# ============================================================
# Cell 11 — ClinicDB loaders
# ============================================================
split = load_emcad_split(CONFIG["clinicdb_root"],
                         os.path.join(CONFIG["output_dir"], "split_clinicdb.json"))
_mk_ds = lambda k: ClinicDBDataset(split[k],
                                   os.path.join(CONFIG["clinicdb_root"], k, "images"),
                                   os.path.join(CONFIG["clinicdb_root"], k, "masks"),
                                   img_size=CONFIG["img_size"], spatial_encoding=CONFIG["spatial_encoding"])
train_ds, val_ds, test_ds = _mk_ds("train"), _mk_ds("val"), _mk_ds("test")

loader_kwargs = dict(
    batch_size         = CONFIG["batch_size"],
    num_workers        = CONFIG["num_workers"],
    pin_memory         = CONFIG["pin_memory"],
    persistent_workers = CONFIG["num_workers"] > 0,
)
if CONFIG["num_workers"] > 0:
    loader_kwargs["prefetch_factor"] = CONFIG["prefetch_factor"]

_g = torch.Generator(); _g.manual_seed(INIT_SEED)
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True, generator=_g, **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

print(f"\n  ClinicDB train: {len(train_ds):>4d} ({len(train_loader)} batches)")
print(f"  ClinicDB val  : {len(val_ds):>4d} ({len(val_loader)} batches)")
print(f"  ClinicDB test : {len(test_ds):>4d} ({len(test_loader)} batches)")


In [ ]:
# ============================================================
# Cell 12 — Hybrid model (MK-UNet + PVM in 8 stages + gate swap) + differential
#           optimizer + LambdaLR scheduler + AMP scaler
# ============================================================
# Model: paper-strict MK-UNet (bottleneck_module=None) with the UltraLight PVM
#        Layer surgically added to encoder2-5 and decoder1-4 via
#        apply_pvm_hybrid(). Style = CONFIG["hybrid_mode"] ("replace"/"insert").
#
# Differential learning rate (CONFIG["lr_mode"]):
#   alpha : PVM layers              -> UltraLight recipe (lr=1e-3, wd=1e-2, cosine)
#           everything else         -> MK-UNet recipe   (lr=1e-4, wd=1e-4, const)
#   beta  : whole modified stages   -> UltraLight recipe
#           pure-CNN stages         -> MK-UNet recipe
#
#   NOTE (REPLACE x alpha == REPLACE x beta): under REPLACE the modified stages
#   contain ONLY PVM params, so the alpha and beta partitions are IDENTICAL ->
#   EXP10 and EXP11 train the same model. For INSERT they differ (beta also puts
#   the kept MKIR params of those stages on the UltraLight recipe).
#
# Scheduler: CosineAnnealingLR(T_max=50, eta_min=1e-5) applied ONLY to the
#   UltraLight group; the MK-UNet group is held CONSTANT (no scheduler), which
#   is what "everything else gets MK-UNet's recipe (... no scheduler)" requires.
#   Implemented as a per-group LambdaLR whose UL lambda is the closed-form
#   cosine (verified bit-identical to torch CosineAnnealingLR) and whose MK
#   lambda is 1.0. T_max/eta_min are CONFIG values — flag for the researcher if
#   Sir prefers T_max = epochs (single decay) instead of the repo's T_max=50.

torch.manual_seed(INIT_SEED); torch.cuda.manual_seed_all(INIT_SEED)
model = build_model(CONFIG).to(DEVICE)   # WACV hybrid + gate swap (Cell 7b)

print(f"Single GPU on {next(model.parameters()).device}")
print(f"Hybrid style : {CONFIG['hybrid_mode'].upper()}  "
      f"(PVM in {', '.join(HYBRID_STAGES)})")

# ── Differential param groups ───────────────────────────────────
ul_params, mk_params, n_ul, n_mk = split_param_groups(model, CONFIG["lr_mode"])

# E4 (augment): alpha gets NO weight decay, so a small final alpha means the
# model chose to ignore the quantum branch, not that weight decay pulled it to 0.
# It keeps the UL learning rate and schedule. Groups 0 and 1 are unchanged.
_nodecay = [p for mm in model.modules() if isinstance(mm, QuantumGroupedAttentionGate)
            and hasattr(mm, "alpha") for p in [mm.alpha]]
_nd_ids  = {id(p) for p in _nodecay}
ul_params = [p for p in ul_params if id(p) not in _nd_ids]
mk_params = [p for p in mk_params if id(p) not in _nd_ids]
_groups = [
    {"params": ul_params, "lr": CONFIG["ul_lr"], "weight_decay": CONFIG["ul_wd"]},  # group 0 = UltraLight
    {"params": mk_params, "lr": CONFIG["mk_lr"], "weight_decay": CONFIG["mk_wd"]},  # group 1 = MK-UNet
]
if _nodecay:
    _groups.append({"params": _nodecay, "lr": CONFIG["ul_lr"], "weight_decay": 0.0})  # group 2 = alpha
optimizer = torch.optim.AdamW(_groups)

# ── Per-group scheduler: cosine on UL group, constant on MK group ─
import math as _math
_T   = CONFIG["ul_cosine_tmax"]
_emi = CONFIG["ul_cosine_eta_min"]
_base= CONFIG["ul_lr"]
def _ul_cosine_lambda(e):
    cos = (1 + _math.cos(_math.pi * e / _T)) / 2
    return (_emi + (_base - _emi) * cos) / _base     # multiplier of base UL lr
def _mk_const_lambda(e):
    return 1.0
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=[_ul_cosine_lambda, _mk_const_lambda] + ([_ul_cosine_lambda] if _nodecay else []))

scaler = torch.amp.GradScaler("cuda")

# ── Param accounting ─────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
pvm_total    = sum(p.numel() for m in model.modules() if isinstance(m, PVMLayer)
                            for p in m.parameters())
cnn_total    = total_params - pvm_total

print(f"\nExperiment : {CONFIG['experiment_name']}")
print(f"Parameters : {total_params:,}  ({total_params/1e6:.3f}M)")
print(f"             PVM params : {pvm_total:,}  ({pvm_total/1e6:.3f}M)")
print(f"             CNN params : {cnn_total:,}  ({cnn_total/1e6:.3f}M)")
gate_total = sum(p.numel() for p in gate_parameters(model, CONFIG["gate_targets"]))
print(f"             Gate params: {gate_total:,}  ({CONFIG['gate_mode']} on {CONFIG['gate_targets']}, "
      f"mixer {count_mixer_params(model)})")
print(f"  (paper-strict MK-UNet baseline = 315,566; REPLACE removes MKIR in the")
print(f"   4 deep stages, INSERT keeps it and adds PVM on top)")
print(f"\nDifferential LR ({CONFIG['lr_mode']}):")
print(f"  UltraLight group : {n_ul:>8,} params  lr={CONFIG['ul_lr']:.0e} wd={CONFIG['ul_wd']:.0e}  "
      f"cosine T_max={_T} eta_min={_emi:.0e}")
print(f"  MK-UNet group    : {n_mk:>8,} params  lr={CONFIG['mk_lr']:.0e} wd={CONFIG['mk_wd']:.0e}  "
      f"constant (no scheduler)")
print(f"Loss       : BceDiceLoss (BCE+Dice 1:1, no boundary weighting, no deep supervision)")
print(f"Optimiser  : AdamW (2 groups)  |  Scheduler: per-group LambdaLR (cosine on UL, const on MK)")
print(f"Grad clip  : {CONFIG['grad_clip']}  |  Multi-scale: {CONFIG['multiscale_train']} {CONFIG['multiscale_rates']}")
print(f"AMP        : enabled  |  cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"RUN_NUMBER : {CONFIG['run_number']}  (seed {CONFIG['seed']})")

if CONFIG["hybrid_mode"] == "replace" and CONFIG["lr_mode"] == "beta":
    print()
    print("⚠ NOTE: REPLACE + beta is mathematically IDENTICAL to REPLACE + alpha")
    print("        (modified stages hold only PVM params). This run duplicates the")
    print("        REPLACE+alpha experiment — you only need to run ONE of them.")

In [ ]:
# ============================================================
# Cell 13 — Resume logic
# ============================================================
ckpt_dir   = CONFIG["checkpoint_dir"]
log_dir    = CONFIG["log_dir"]
output_dir = CONFIG["output_dir"]
grad_clip  = CONFIG["grad_clip"]

CSV_HEADER = [
    "epoch", "train_loss",
    "val_dice", "val_iou", "val_sensitivity", "val_specificity",
    "val_precision", "val_accuracy",
    "lr", "lr_mk", "epoch_time_sec", "gpu_mem_peak_gb",
]
# Note: test metrics are NOT logged per epoch. Test set is evaluated ONCE
# after training, on the best-val checkpoint (cell 16). This avoids
# any test-set peeking during model development.
csv_path = os.path.join(log_dir, "training_log.csv")


def find_resume_checkpoint(kaggle_input_dir, local_ckpt_dir):
    if kaggle_input_dir and os.path.isdir(kaggle_input_dir):
        for preferred in ("last_model.pth", "best_model.pth"):
            for root, _, files in os.walk(kaggle_input_dir):
                if preferred in files:
                    return os.path.join(root, preferred)
    for preferred in ("last_model.pth", "best_model.pth"):
        p = os.path.join(local_ckpt_dir, preferred)
        if os.path.exists(p):
            return p
    return None


def load_history_from_csv(csv_file):
    rows = []
    with open(csv_file, newline="") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append({k: r.get(k, "") for k in CSV_HEADER})
    return rows


start_epoch                 = 0
best_val_dice               = 0.0
best_epoch                  = 0
training_history            = []
total_elapsed_seconds       = 0.0

if RESUME:
    ckpt_path = find_resume_checkpoint(KAGGLE_MODEL_INPUT_DIR, ckpt_dir)
    if ckpt_path is None:
        print("RESUME=True but no checkpoint found — starting from scratch.")
    else:
        print(f"Resuming from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

        ckpt_cfg = ckpt.get("config", {})
        if ckpt_cfg.get("spatial_encoding") not in (None, CONFIG["spatial_encoding"]) \
           or ckpt_cfg.get("in_channels") not in (None, CONFIG["in_channels"]):
            raise RuntimeError(
                f"Checkpoint config mismatch.\n"
                f"  Checkpoint: spatial_encoding={ckpt_cfg.get('spatial_encoding')!r}, "
                f"in_channels={ckpt_cfg.get('in_channels')!r}\n"
                f"  Current:    spatial_encoding={CONFIG['spatial_encoding']!r}, "
                f"in_channels={CONFIG['in_channels']!r}"
            )

        m = model.module if hasattr(model, "module") else model
        m.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        best_val_dice         = float(ckpt.get("best_val_dice", 0.0))
        best_epoch            = int(ckpt.get("best_epoch", 0))
        start_epoch           = int(ckpt.get("epoch", 0))
        total_elapsed_seconds = float(ckpt.get("total_elapsed_seconds", 0.0))

        embedded = list(ckpt.get("training_history", []))
        if embedded:
            training_history = embedded
            print(f"  History (embedded) : {len(embedded)} rows")

        print(f"  Epoch reached      : {start_epoch} / {CONFIG['epochs']}")
        print(f"  Best val Dice      : {best_val_dice:.4f}  @ epoch {best_epoch}")
        print(f"  Elapsed (cum.)     : {total_elapsed_seconds/60:.1f} min")
else:
    for old in ["last_model.pth", "best_model.pth"]:
        p = os.path.join(ckpt_dir, old)
        if os.path.exists(p):
            os.remove(p)
    for ep in glob.glob(os.path.join(ckpt_dir, "epoch_*.pth")):
        os.remove(ep)
    print("Fresh training (RESUME=False) — stale local checkpoints cleared.")

# Write CSV with header + any prior rows
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(CSV_HEADER)
    for row in training_history:
        w.writerow([row.get(k, "") for k in CSV_HEADER])
    f.flush()
    os.fsync(f.fileno())
print(f"CSV log: {csv_path}  ({len(training_history)} prior rows)")

In [ ]:
# ============================================================
# Cell 14 — Training loop (hybrid, val-only per epoch)
# ============================================================
# Eval policy: validation only during training; the test set is held out until
# Cell 16 runs ONCE on the best-val checkpoint. Model selection = highest val Dice.
#
# Differential LR: two AdamW groups. The UltraLight group follows a cosine
# schedule (stepped once per epoch); the MK-UNet group is held constant. Both
# group LRs are logged per epoch (lr = UL, lr_mk = MK).
#
# ⚠ KAGGLE SESSION RECOVERY: click "Save Version -> Quick Save" every ~30 epochs.
#   Files in /kaggle/working are lost on timeout/crash unless saved. CSV is
#   flushed+fsynced every epoch.

print(f"\n{'='*80}")
print(f"STARTING TRAINING — {CONFIG['experiment_name']}")
print(f"  hybrid {CONFIG['hybrid_mode'].upper()} + {CONFIG['lr_mode']} differential LR, "
      f"BceDiceLoss, {CONFIG['epochs']} epochs, batch {CONFIG['batch_size']}, 256x256")
print(f"  UL group: lr={CONFIG['ul_lr']:.0e} wd={CONFIG['ul_wd']:.0e} cosine T_max={CONFIG['ul_cosine_tmax']}")
print(f"  MK group: lr={CONFIG['mk_lr']:.0e} wd={CONFIG['mk_wd']:.0e} constant")
print(f"  multi-scale rates: {CONFIG['multiscale_rates']}")
print(f"  starting at epoch index {start_epoch}, target {CONFIG['epochs']}")
print(f"  EVAL POLICY: val every epoch, test ONCE after training")
print(f"{'='*80}")

session_start = datetime.now()


def make_checkpoint(extra=None):
    state_dict = (model.module if hasattr(model, "module") else model).state_dict()
    ckpt = {
        "epoch"                  : epoch + 1,
        "model_state_dict"       : state_dict,
        "optimizer_state_dict"   : optimizer.state_dict(),
        "scheduler_state_dict"   : scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict"      : scaler.state_dict(),
        "best_val_dice"          : best_val_dice,
        "best_epoch"             : best_epoch,
        "training_history"       : training_history,
        "total_elapsed_seconds"  : total_elapsed_seconds,
        "config"                 : CONFIG,
    }
    if extra:
        ckpt.update(extra)
    return ckpt


for epoch in range(start_epoch, CONFIG["epochs"]):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    epoch_start = time.time()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, DEVICE, scaler, grad_clip,
        multiscale_train=CONFIG["multiscale_train"],
        multiscale_rates=CONFIG["multiscale_rates"],
        img_size=CONFIG["img_size"],
    )

    val_m, _ = evaluate_paper_strict(model, val_loader, DEVICE, desc="  Val ")

    lr_ul = optimizer.param_groups[0]["lr"]
    lr_mk = optimizer.param_groups[1]["lr"]
    epoch_time = time.time() - epoch_start
    total_elapsed_seconds += epoch_time
    gpu_mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9

    is_best = val_m["dice"] > best_val_dice
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}   "
          f"[{epoch_time:6.1f}s, peak {gpu_mem_peak_gb:.2f} GB]")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Dice   : {val_m['dice']:.4f}  (best {max(best_val_dice, val_m['dice']):.4f}"
          f"{'  ◄ NEW BEST' if is_best else ''})")
    print(f"  Val IoU    : {val_m['iou']:.4f}  |  Val Sens: {val_m['sensitivity']:.4f}  "
          f"|  Val Spec: {val_m['specificity']:.4f}")
    print(f"  Val Prec   : {val_m['precision']:.4f}  |  Val Acc : {val_m['accuracy']:.4f}")
    print(f"  LR (UL/MK) : {lr_ul:.2e} / {lr_mk:.2e}")
    print(f"  Cum. time  : {total_elapsed_seconds/60:.1f} min")
    print("-"*80)

    row = {
        "epoch"            : epoch + 1,
        "train_loss"       : f"{train_loss:.4f}",
        "val_dice"         : f"{val_m['dice']:.4f}",
        "val_iou"          : f"{val_m['iou']:.4f}",
        "val_sensitivity"  : f"{val_m['sensitivity']:.4f}",
        "val_specificity"  : f"{val_m['specificity']:.4f}",
        "val_precision"    : f"{val_m['precision']:.4f}",
        "val_accuracy"     : f"{val_m['accuracy']:.4f}",
        "lr"               : f"{lr_ul:.6f}",
        "lr_mk"            : f"{lr_mk:.6f}",
        "epoch_time_sec"   : f"{epoch_time:.2f}",
        "gpu_mem_peak_gb"  : f"{gpu_mem_peak_gb:.3f}",
    }
    training_history.append(row)

    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow([row[k] for k in CSV_HEADER])
        f.flush()
        os.fsync(f.fileno())

    if is_best:
        best_val_dice = val_m["dice"]
        best_epoch    = epoch + 1
        torch.save(make_checkpoint(extra={"val_metrics": val_m}),
                   os.path.join(ckpt_dir, "best_model.pth"))
        print(f"  ✓ best_model.pth saved (Val Dice {best_val_dice:.4f})")

    if (epoch + 1) % 10 == 0:
        torch.save(make_checkpoint(), os.path.join(ckpt_dir, f"epoch_{epoch+1}.pth"))

    torch.save(make_checkpoint(), os.path.join(ckpt_dir, "last_model.pth"))

    # Step the per-group scheduler AFTER logging (cosine on UL, constant on MK).
    if scheduler is not None:
        scheduler.step()

session_time = datetime.now() - session_start
print(f"\n{'='*80}")
print(f"TRAINING COMPLETE — {CONFIG['experiment_name']}")
print(f"  Best Val Dice : {best_val_dice:.4f} @ Epoch {best_epoch}")
print(f"  This session  : {session_time}")
print(f"  Total time    : {total_elapsed_seconds/60:.1f} min")
print(f"  ➜ Test set evaluated next in Cell 16 (one-shot, paper-strict).")
print(f"{'='*80}")

In [ ]:
# ============================================================
# Cell 15 — Training curves
# ============================================================
log_df = pd.read_csv(csv_path)
for col in log_df.columns:
    if col != "epoch":
        log_df[col] = pd.to_numeric(log_df[col], errors="coerce")
    else:
        log_df[col] = pd.to_numeric(log_df[col], errors="coerce").astype("Int64")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].plot(log_df["epoch"], log_df["train_loss"], color="steelblue", label="Train")
axes[0,0].set_title("Train Loss (BCE+Dice)"); axes[0,0].set_xlabel("Epoch")
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(log_df["epoch"], log_df["val_dice"],  color="green",  label="Val Dice")
axes[0,1].axhline(y=best_val_dice, color="green", linestyle="--",
                  label=f"Best Val {best_val_dice:.4f}", alpha=0.5)
axes[0,1].set_title("Validation Dice per epoch"); axes[0,1].set_xlabel("Epoch")
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(log_df["epoch"], log_df["val_iou"], color="purple")
axes[0,2].set_title("Val IoU"); axes[0,2].set_xlabel("Epoch")
axes[0,2].grid(alpha=0.3)

axes[1,0].plot(log_df["epoch"], log_df["lr"], color="black", label="UL group (cosine)")
if "lr_mk" in log_df.columns:
    axes[1,0].plot(log_df["epoch"], log_df["lr_mk"], color="orange", label="MK group (const)")
axes[1,0].set_title("Learning Rate (differential)"); axes[1,0].set_xlabel("Epoch")
axes[1,0].set_yscale("log"); axes[1,0].legend(fontsize=8); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(log_df["epoch"], log_df["epoch_time_sec"], color="teal")
axes[1,1].set_title("Epoch time (s)"); axes[1,1].set_xlabel("Epoch")
axes[1,1].grid(alpha=0.3)

axes[1,2].plot(log_df["epoch"], log_df["gpu_mem_peak_gb"], color="crimson")
axes[1,2].set_title("GPU peak memory (GB)"); axes[1,2].set_xlabel("Epoch")
axes[1,2].grid(alpha=0.3)

plt.suptitle(f"{CONFIG['experiment_name']} | "
             f"Best Val Dice={best_val_dice:.4f} @ Epoch {best_epoch}")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved.")

In [ ]:
# ============================================================
# Cell 16 — ClinicDB test set evaluation (best checkpoint, paper-strict)
# ============================================================
print("="*80)
print(f"ClinicDB TEST EVALUATION — {CONFIG['experiment_name']}")
print("="*80)

best_ckpt = torch.load(os.path.join(ckpt_dir, "best_model.pth"),
                       map_location=DEVICE, weights_only=False)
m = model.module if hasattr(model, "module") else model
m.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded best model: epoch {best_ckpt['epoch']}  "
      f"val_dice={best_ckpt.get('val_metrics',{}).get('dice', best_val_dice):.4f}")

t0 = time.time()
test_m, test_preds = evaluate_paper_strict(model, test_loader, DEVICE, desc="ClinicDB test")
test_elapsed = time.time() - t0

print(f"\n{'='*80}")
print(f"ClinicDB TEST RESULTS  ({CONFIG['experiment_name']}, paper-strict eval)")
print("="*80)
for k, v in test_m.items():
    print(f"  {k:<14}: {v:.4f}")
print(f"  {'eval_time':<14}: {test_elapsed:.2f} s")
print("="*80)

test_summary = {
    "experiment_name" : CONFIG["experiment_name"],
    "spatial_encoding": CONFIG["spatial_encoding"],
    "in_channels"     : CONFIG["in_channels"],
    "test_metrics"    : test_m,
    "best_epoch"      : best_ckpt["epoch"],
    "n_samples"       : len(test_ds),
    "eval_protocol"   : "paper_strict",
    "eval_time_total_s" : round(test_elapsed, 3),
    "config"          : CONFIG,
}
with open(os.path.join(output_dir, "clinicdb_test_results.json"), "w") as f:
    json.dump(test_summary, f, indent=4)
print("ClinicDB test results saved.")


In [ ]:
# ============================================================
# Cell 17 — Per-image test metrics (for Wilcoxon across runs) + results record
# ============================================================
# Saves per-image Dice/IoU with file names, so E2 vs E0 and E2 vs E3 can be
# compared with a paired Wilcoxon test later. Same paper-strict steps as
# evaluate_paper_strict().

@torch.no_grad()
def per_image_scores(model, dataset, loader, device):
    model.eval()
    rows, k = [], 0
    for imgs, masks, shapes in tqdm(loader, desc="per-image", leave=False):
        pred = model(imgs.to(device))
        pred = pred[0] if isinstance(pred, list) else pred
        masks = masks.to(device).float()
        for b in range(imgs.size(0)):
            H, W = int(shapes[b][0]), int(shapes[b][1])
            p = torch.sigmoid(F.interpolate(pred[b:b+1], size=(H, W), mode="bilinear",
                                            align_corners=False)).squeeze()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            t = (F.interpolate(masks[b:b+1], size=(H, W), mode="nearest").squeeze() >= 0.2).float()
            m = per_image_metrics((p >= 0.5).float(), t)
            rows.append({"image": dataset.pairs[k][0], "dice": m["dice"], "iou": m["iou"]})
            k += 1
    return pd.DataFrame(rows)

per_img = per_image_scores(model, test_ds, test_loader, DEVICE)
per_img.to_csv(os.path.join(output_dir, "per_image_test.csv"), index=False)
print(f"Per-image scores saved ({len(per_img)} images). Mean Dice {per_img.dice.mean():.4f}")

gm = getattr(model, CONFIG["gate_targets"][0])
results = {
    "experiment_name" : CONFIG["experiment_name"],
    "gate_mode"       : CONFIG["gate_mode"],
    "gate_targets"    : CONFIG["gate_targets"],
    "qag_layers"      : CONFIG["qag_layers"],
    "init_seed"       : CONFIG["init_seed"],
    "run_number"      : CONFIG["run_number"],
    "test_metrics"    : test_m,
    "best_epoch"      : best_epoch,
    "best_val_dice"   : best_val_dice,
    "params_total"    : sum(p.numel() for p in model.parameters()),
    "params_gate"     : sum(p.numel() for p in gate_parameters(model, CONFIG["gate_targets"])),
    "params_mixer"    : count_mixer_params(model),
    "alpha_final"     : float(gm.alpha.item()) if hasattr(gm, "alpha") else None,
    "train_minutes"   : round(total_elapsed_seconds / 60, 1),
    "gpu"             : torch.cuda.get_device_name(0),
}
with open(os.path.join(output_dir, "results_record.json"), "w") as f:
    json.dump(results, f, indent=4)
print(json.dumps({k: v for k, v in results.items() if k != "test_metrics"}, indent=2))
if results["alpha_final"] is not None:
    print(f"\naugment mode: final alpha = {results['alpha_final']:.4f} "
          f"(near 0 means the model ignored the quantum branch)")


In [ ]:
# ============================================================
# Cell 19 — Prediction visualisations (ClinicDB test)
# ============================================================
def _plot_grid(dataset, preds, n, title, save_to):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    n = min(n, len(preds), len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(9, n*3))
    fig.suptitle(title, fontsize=12)
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        img_full, mask, _shape = dataset[i]
        rgb  = (img_full[:3] * std + mean).clamp(0,1).permute(1,2,0).numpy()
        pred = (preds[i].squeeze() > 0.5).numpy()
        axes[i,0].imshow(rgb);                          axes[i,0].axis("off")
        axes[i,1].imshow(mask.squeeze(), cmap="gray");  axes[i,1].axis("off")
        axes[i,2].imshow(pred, cmap="gray");            axes[i,2].axis("off")
        if i == 0:
            for ax, t in zip(axes[0], ["Image","Ground Truth","Prediction"]):
                ax.set_title(t)
    plt.tight_layout()
    plt.savefig(save_to, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_to}")


_plot_grid(test_ds, test_preds, 4,
           f"{CONFIG['experiment_name']} | ClinicDB test predictions",
           os.path.join(output_dir, "clinicdb_predictions.png"))


In [ ]:
# ============================================================
# Cell 20 — Run summary + output inventory
# ============================================================
print(f"\n{'='*80}")
print(f"RUN SUMMARY  —  {CONFIG['experiment_name']}")
print(f"{'='*80}")
print(f"  Spatial encoding       : {CONFIG['spatial_encoding']}")
print(f"  In channels            : {CONFIG['in_channels']}")
print(f"  Image size             : {CONFIG['img_size']}x{CONFIG['img_size']}")
print(f"  Epochs completed       : {len(training_history)} / {CONFIG['epochs']}")
print(f"  Best Val Dice          : {best_val_dice:.4f}  @ epoch {best_epoch}")
print(f"  Gate                   : {CONFIG['gate_mode']} on {CONFIG['gate_targets']} (L={CONFIG['qag_layers']})")
print(f"  ClinicDB Test Dice     : {test_m['dice']:.4f}  ◄ HEADLINE METRIC (one-shot eval)")
print(f"  ClinicDB Test IoU      : {test_m['iou']:.4f}")
print(f"  Cumulative train time  : {total_elapsed_seconds/60:.1f} min")

print(f"\n{'='*80}")
print(f"FILES UNDER /kaggle/working  (slug: {CONFIG['slug']})")
print(f"{'='*80}")
files = [
    os.path.join(ckpt_dir,   "best_model.pth"),
    os.path.join(ckpt_dir,   "last_model.pth"),
    os.path.join(log_dir,    "training_log.csv"),
    os.path.join(output_dir, "config.json"),
    os.path.join(output_dir, "split_clinicdb.json"),
    os.path.join(output_dir, "clinicdb_test_results.json"),
    os.path.join(output_dir, "results_record.json"),
    os.path.join(output_dir, "per_image_test.csv"),
    os.path.join(output_dir, "training_curves.png"),
    os.path.join(output_dir, "clinicdb_predictions.png"),
]
for p in files:
    flag = "✓" if os.path.exists(p) else "✗"
    size = f"{os.path.getsize(p)/1e6:7.2f} MB" if os.path.exists(p) else "    -    "
    print(f"  {flag}  {size}   {p}")

print(f"\n{'='*80}")
print("⚠ KAGGLE OUTPUT PERSISTENCE")
print(f"{'='*80}")
print("If you ran this end-to-end, click 'Save Version → Quick Save' to commit")
print("the outputs to the notebook's Output tab. Otherwise, files in /kaggle/working")
print("will be discarded when the session ends.")
print("="*80)
